# Notebook 4: Statistical Analysis of Experiment Results

This notebook loads the data from a completed experiment run (identified by its `experiment_id`) and performs a comprehensive statistical analysis. It replicates the functionality of the `scripts/export_reporting.py` script in an interactive format.

**Analysis Steps:**
1. **Load Data**: Fetch the raw logs from the `policy_evaluation_log` table for the specified experiment.
2. **Compute Metrics**: Calculate key performance indicators (KPIs) like Average Process Time (APT), quantiles (Q90), Conditional Value-at-Risk (CVaR), and Miss Rate for various policy and scenario groups.
3. **Significance Testing**: Perform Welch's t-tests to compare the performance of different policies and apply the Holm-Bonferroni correction for multiple comparisons.
4. **Generate Summaries**: Create summary tables that are ready for reporting.

**Instructions:**
- Paste the `experiment_id` you obtained from the previous notebook (`03_offline_replay...`) into the designated cell below.

In [ ]:
import sys
import os
import pandas as pd

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.analytics import OfflineReplayAnalyzer, TableExporter
from pathlib import Path

# Configure pandas for better display
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)

### 1. Set Experiment ID and Initialize Analyzer

👇 **ACTION REQUIRED**: Paste your `experiment_id` from the previous notebook here.

In [ ]:
# PASTE YOUR EXPERIMENT ID HERE
experiment_id = "<PASTE YOUR EXPERIMENT ID HERE>"

if "<PASTE" in experiment_id:
    print("⚠️ Please replace the placeholder with your actual experiment ID.")
else:
    print(f"Analyzing Experiment ID: {experiment_id}")
    analyzer = OfflineReplayAnalyzer(experiment_id)


### 2. Load and Prepare Data

In [ ]:
try:
    analyzer.load_data()
    analyzer.prepare_data()
    print("Data loaded and prepared successfully.")
    display(analyzer.df_log.head())
    print(f"\nTotal log entries: {len(analyzer.df_log)}")
except Exception as e:
    print(f"❌ Failed to load data. Have you pasted the correct experiment ID? Error: {e}")

### 3. Analysis of Q4 (Parking Scenario)

The Q4 scenario is often used for primary policy comparisons due to its complexity (involving parking occupancy). Here we compute the main performance metrics for each policy group within this scenario.

In [ ]:
df_q4_metrics = analyzer._analyze_q4_performance()
print("--- Q4 Performance Metrics by Policy ---")
display(df_q4_metrics)

### 4. Statistical Significance Testing (E1-E5)

This performs Welch's t-tests to compare key policy pairs and calculates Cohen's d for effect size. The p-values are then adjusted using the Holm-Bonferroni method to control the family-wise error rate.

In [ ]:
df_tests, p_values = analyzer._run_e1_e5_tests()
analyzer.results['e1_e5_tests'] = df_tests # Store before correction
analyzer._apply_holm_bonferroni(p_values)

print("--- Statistical Test Results (E1-E5 Experiments) ---")
display(analyzer.results['e1_e5_tests'])

### 5. Overall Policy Summary

This table provides a high-level summary of each policy's performance across all scenarios, including the `switch_rate`, which measures recommendation stability.

In [ ]:
df_policy_summary = analyzer._generate_policy_summary()
print("--- Overall Policy Summary (All Scenarios) ---")
display(df_policy_summary.sort_values('APT_mean'))

### 6. Export All Results

Finally, we can run the full pipeline function to populate all result tables and export them to an Excel file and individual CSVs in the `output/` directory.

In [ ]:
print("Running the full analysis and export pipeline...")
output_dir = Path(project_root) / 'output'
analyzer.run_full_pipeline(output_dir=str(output_dir))
print(f"\n✅ All results have been exported to the '{output_dir.name}' directory.")